## Hoeffding Planner API (Qiskit-friendly)

* One circuit shot produces an outcome $X \in [a,b]$.
* We want an estimator $\hat{\mu_N}$ of $\mu=E[X]$ such that

$$
Pr( | | )
$$

In [4]:
!pip install -q qiskit qiskit-aer qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 69.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 59.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.4/377.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 47.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.1 MB/s eta 0:00:00


In [1]:
# hoeffding_planner.py
from dataclasses import dataclass
from math import log, ceil

@dataclass
class HoeffdingPlanner:
    """Generic Hoeffding shot planner for a bounded random variable X in [a, b]."""
    epsilon_stat: float  # tolerance on |hat(mu) - mu| (for X itself)
    delta: float         # failure probability
    a: float             # lower bound of X
    b: float             # upper bound of X

    def planned_shots(self) -> int:
        if self.epsilon_stat <= 0:
            raise ValueError("epsilon_stat must be > 0")
        if not (0 < self.delta < 1):
            raise ValueError("delta must be in (0,1)")
        R = self.b - self.a
        n = (R**2 / (2 * self.epsilon_stat**2)) * log(2.0 / self.delta)
        return int(ceil(n))


In [2]:
# swap_planning.py
# from hoeffding_planner import HoeffdingPlanner

def plan_shots_for_swap_fidelity(epsilon_F: float, delta: float) -> int:
    """
    Plan shots so that the SWAP-test fidelity estimate F_hat satisfies
    |F_hat - F| <= epsilon_F with failure probability <= delta,
    assuming we estimate F via the ancilla Z expectation (±1 outcomes).
    """
    planner = HoeffdingPlanner(
        epsilon_stat=epsilon_F,  # because F = E[Z]
        delta=delta,
        a=-1.0,
        b=+1.0,
    )
    return planner.planned_shots()


In [5]:
# swap_circuit.py
from qiskit import QuantumCircuit

def swap_test_1qubit(theta1: float, theta2: float) -> QuantumCircuit:
    """
    3-qubit SWAP test:
      - qubit 0: ancilla
      - qubit 1: |psi> prepared with Ry(theta1)
      - qubit 2: |phi> prepared with Ry(theta2)
    """
    qc = QuantumCircuit(3)

    # Prepare |psi> on qubit 1
    qc.ry(theta1, 1)

    # Prepare |phi> on qubit 2
    qc.ry(theta2, 2)

    # SWAP test gadget
    qc.h(0)                       # Hadamard on ancilla
    qc.cswap(0, 1, 2)            # controlled-SWAP between qubits 1 and 2
    qc.h(0)                       # Hadamard on ancilla again

    # No measurement here: Estimator will conceptually measure Pauli Z on qubit 0
    return qc


In [17]:
%pip install -q qiskit[visualization]


In [18]:
# Verify visualization dependencies are available
try:
    import pylatexenc
    print(f"✓ pylatexenc version: {pylatexenc.__version__}")
except ImportError:
    print("✗ pylatexenc not found")

try:
    import matplotlib
    print(f"✓ matplotlib version: {matplotlib.__version__}")
except ImportError:
    print("✗ matplotlib not found")

# Try importing qiskit visualization to check if it's ready
try:
    from qiskit.visualization import circuit_drawer
    print("✓ Qiskit visualization module loaded successfully")
except Exception as e:
    print(f"✗ Qiskit visualization error: {e}")


✓ pylatexenc version: 2.10
✓ matplotlib version: 3.10.0
✓ Qiskit visualization module loaded successfully


In [21]:
# demo_swap_hoeffding.py
import math

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.transpiler import generate_preset_pass_manager

# from swap_circuit import swap_test_1qubit
# from swap_planning import plan_shots_for_swap_fidelity

# 1. Choose the pair of states
theta1 = 0.3
theta2 = 0.8

qc = swap_test_1qubit(theta1, theta2)

# 2. Build an observable: Pauli Z on the ancilla (qubit 0)
# For 3 qubits, ancilla is qubit 0, so we want "ZII"
observable = SparsePauliOp(["ZII"], coeffs=[1.0])

# 3. Define a simple depolarizing noise model on 1-qubit gates
noise_model = NoiseModel()
p_1q = 0.01  # 1% depolarizing error on all 1q gates
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(p_1q, 1),
    ["ry", "rz", "sx", "x", "h"],  # adjust if needed
)

# 4. Transpile for AerSimulator (optional but realistic)
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
qc_isa = pm.run(qc)

# 5. Plan shots using Hoeffding for fidelity error
epsilon_F = 0.02   # want |F_hat - F| <= 0.02
delta = 0.01       # with probability at least 99%

shots = plan_shots_for_swap_fidelity(epsilon_F, delta)
print(f"Planned shots (Hoeffding) for fidelity: {shots}")

options = {
    "backend_options": {
        "noise_model": noise_model,
        "seed_simulator": 1234,
    },
    "run_options": {
        "shots": shots,
    },
}

estimator = AerEstimator(options=options)

# EstimatorV2 "pub": (circuit, observable, [parameters])
pub = (qc_isa, observable, [])

job = estimator.run([pub])
result = job.result()
pub_result = result[0]

F_hat = float(pub_result.data.evs)   # estimated E[Z] on ancilla
F_ideal = math.cos((theta1 - theta2)/2)**2

print(f"Ideal fidelity: {F_ideal:.4f}")
print(f"Noisy estimated fidelity F_hat: {F_hat:.4f}")
print(f"|F_hat - F_ideal| = {abs(F_hat - F_ideal):.4f}")
print(f"Metadata: {pub_result.metadata}")


Planned shots (Hoeffding) for fidelity: 26492
Ideal fidelity: 0.9388
Noisy estimated fidelity F_hat: 0.8186
|F_hat - F_ideal| = 0.1202
Metadata: {'target_precision': 0.0, 'circuit_metadata': {}, 'simulator_metadata': {'time_taken_parameter_binding': 9.785e-06, 'time_taken_execute': 1.170449501, 'omp_enabled': True, 'max_gpu_memory_mb': 0, 'max_memory_mb': 12975, 'parallel_experiments': 1}}


In [22]:
# demo_swap_hoeffding.py

import math
from dataclasses import dataclass
from math import log, ceil

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.transpiler import generate_preset_pass_manager

# ---------------------------------------------------------------------
# 1. SWAP-test circuit (3 qubits: ancilla + 2 data qubits)
# ---------------------------------------------------------------------

def swap_test_1qubit(theta1: float, theta2: float) -> QuantumCircuit:
    """
    3-qubit SWAP test:
      - qubit 0: ancilla
      - qubit 1: |psi> = Ry(theta1)|0>
      - qubit 2: |phi> = Ry(theta2)|0>
    """
    qc = QuantumCircuit(3)

    # Prepare |psi> on qubit 1
    qc.ry(theta1, 1)

    # Prepare |phi> on qubit 2
    qc.ry(theta2, 2)

    # SWAP-test gadget
    qc.h(0)
    qc.cswap(0, 1, 2)
    qc.h(0)

    # No classical measurement here; Estimator measures Pauli Z on ancilla.
    return qc

# ---------------------------------------------------------------------
# 2. Generic Hoeffding planner for bounded X in [a, b]
# ---------------------------------------------------------------------

@dataclass
class HoeffdingPlanner:
    """Hoeffding shot planner for a bounded random variable X in [a, b]."""
    epsilon_stat: float  # tolerance on |hat(mu) - mu| for X itself
    delta: float         # failure probability
    a: float             # lower bound
    b: float             # upper bound

    def planned_shots(self) -> int:
        if self.epsilon_stat <= 0:
            raise ValueError("epsilon_stat must be > 0")
        if not (0 < self.delta < 1):
            raise ValueError("delta must be in (0,1)")

        R = self.b - self.a
        n = (R**2 / (2 * self.epsilon_stat**2)) * log(2.0 / self.delta)
        return int(ceil(n))

# ---------------------------------------------------------------------
# 3. Convenience: plan shots specifically for SWAP fidelity
#    We treat ancilla Z outcome as X in {-1,+1}, so F = E[X]
# ---------------------------------------------------------------------

def plan_shots_for_swap_fidelity(epsilon_F: float, delta: float) -> int:
    """
    Plan shots so that the SWAP-test fidelity estimate F_hat satisfies
    |F_hat - F_device| <= epsilon_F with failure probability <= delta,
    where F_device is the device's true (noisy) fidelity.
    """
    planner = HoeffdingPlanner(
        epsilon_stat=epsilon_F,  # F = E[Z] directly
        delta=delta,
        a=-1.0,
        b=+1.0,
    )
    return planner.planned_shots()

# ---------------------------------------------------------------------
# 4. Demo: compare ideal, noisy reference, and Hoeffding-planned estimate
# ---------------------------------------------------------------------

# Choose the pair of states
theta1 = 0.3
theta2 = 0.8

qc = swap_test_1qubit(theta1, theta2)

# Observable: Pauli Z on ancilla (qubit 0) for a 3-qubit circuit -> "ZII"
observable = SparsePauliOp(["ZII"], coeffs=[1.0])

# Noiseless ideal fidelity (analytic)
F_ideal = math.cos((theta1 - theta2) / 2) ** 2

# Build a simple depolarizing noise model on 1-qubit gates
noise_model = NoiseModel()
p_1q = 0.01  # 1% depolarizing error on all 1q gates
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(p_1q, 1),
    ["ry", "rz", "sx", "x", "h"],
)

# Transpile for AerSimulator
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
qc_isa = pm.run(qc)

# ---------- High-precision NOISY reference (what the device "really" does) -----

big_options = {
    "backend_options": {
        "noise_model": noise_model,
        "seed_simulator": 9999,
    },
    "run_options": {
        "shots": 1_000_000,  # huge for tight reference on F_device
    },
}
big_estimator = AerEstimator(options=big_options)
pub = (qc_isa, observable, [])
big_job = big_estimator.run([pub])
big_result = big_job.result()[0]
F_noisy_ref = float(big_result.data.evs)

# ---------- Hoeffding-planned run ---------------------------------------------

epsilon_F = 0.02   # want |F_hat - F_device| <= 0.02
delta = 0.01       # with probability >= 99%

shots = plan_shots_for_swap_fidelity(epsilon_F, delta)
print(f"Planned shots (Hoeffding) for fidelity: {shots}")

options = {
    "backend_options": {
        "noise_model": noise_model,
        "seed_simulator": 1234,
    },
    "run_options": {
        "shots": shots,
    },
}
estimator = AerEstimator(options=options)
job = estimator.run([pub])
result = job.result()
pub_result = result[0]

F_hat = float(pub_result.data.evs)   # estimated E[Z] on ancilla

# ---------- Report all three levels -------------------------------------------

print("\n=== SWAP-test fidelity under noisy channel ===")
print(f"Ideal (noiseless) fidelity F_ideal:    {F_ideal:.4f}")
print(f"Noisy reference fidelity F_device:     {F_noisy_ref:.4f}  (1e6 shots)")
print(f"Hoeffding-planned estimate F_hat:      {F_hat:.4f}  ({shots} shots)")

print(f"\nStatistical error wrt device: |F_hat - F_device| = {abs(F_hat - F_noisy_ref):.4f}")
print(f"Hardware bias wrt ideal:      |F_device - F_ideal| = {abs(F_noisy_ref - F_ideal):.4f}")
print(f"Total gap to ideal:           |F_hat - F_ideal|   = {abs(F_hat - F_ideal):.4f}")

print(f"\nMetadata (planned run): {pub_result.metadata}")


Planned shots (Hoeffding) for fidelity: 26492

=== SWAP-test fidelity under noisy channel ===
Ideal (noiseless) fidelity F_ideal:    0.9388
Noisy reference fidelity F_device:     0.8178  (1e6 shots)
Hoeffding-planned estimate F_hat:      0.8186  (26492 shots)

Statistical error wrt device: |F_hat - F_device| = 0.0008
Hardware bias wrt ideal:      |F_device - F_ideal| = 0.1210
Total gap to ideal:           |F_hat - F_ideal|   = 0.1202

Metadata (planned run): {'target_precision': 0.0, 'circuit_metadata': {}, 'simulator_metadata': {'time_taken_parameter_binding': 1.3933e-05, 'time_taken_execute': 2.097844137, 'omp_enabled': True, 'max_gpu_memory_mb': 0, 'max_memory_mb': 12975, 'parallel_experiments': 1}}


In [23]:
# ---------------------------------------------------------------------
# 5. Validation: Run multiple experiments to check Hoeffding coverage
# ---------------------------------------------------------------------

import numpy as np
from tqdm import tqdm

# Parameters for validation
n_trials = 1000  # number of independent experiments
epsilon_F = 0.02   # same tolerance as before
delta = 0.01       # same failure probability

# Use the same circuit, observable, and noise model from above
# (qc_isa, observable, noise_model, F_noisy_ref already defined)

shots = plan_shots_for_swap_fidelity(epsilon_F, delta)
print(f"Running {n_trials} independent experiments with {shots} shots each...")
print(f"Expected failure rate (delta): {delta:.1%}")
print(f"Tolerance (epsilon_F): {epsilon_F:.4f}\n")

# Store results
errors = []
failures = 0

# Run multiple trials with different random seeds
for trial in tqdm(range(n_trials), desc="Trials"):
    # Each trial uses a different seed for randomness
    trial_options = {
        "backend_options": {
            "noise_model": noise_model,
            "seed_simulator": trial,  # different seed per trial
        },
        "run_options": {
            "shots": shots,
        },
    }
    trial_estimator = AerEstimator(options=trial_options)
    trial_job = trial_estimator.run([pub])
    trial_result = trial_job.result()[0]
    F_hat_trial = float(trial_result.data.evs)
    
    # Compute statistical error
    error = abs(F_hat_trial - F_noisy_ref)
    errors.append(error)
    
    # Check if bound is violated
    if error > epsilon_F:
        failures += 1

# Compute empirical failure rate
empirical_failure_rate = failures / n_trials

# Statistics
errors_array = np.array(errors)
mean_error = np.mean(errors_array)
max_error = np.max(errors_array)
p95_error = np.percentile(errors_array, 95)
p99_error = np.percentile(errors_array, 99)

# Report results
print("\n" + "="*60)
print("HOEFFDING BOUND VALIDATION RESULTS")
print("="*60)
print(f"Total trials:                    {n_trials}")
print(f"Shots per trial:                 {shots}")
print(f"\nBound violations:                {failures} / {n_trials}")
print(f"Empirical failure rate:          {empirical_failure_rate:.4f} ({empirical_failure_rate:.2%})")
print(f"Theoretical failure rate (delta): {delta:.4f} ({delta:.2%})")
print(f"\nStatistical error statistics:")
print(f"  Mean error:                    {mean_error:.6f}")
print(f"  Max error:                     {max_error:.6f}")
print(f"  95th percentile error:         {p95_error:.6f}")
print(f"  99th percentile error:         {p99_error:.6f}")
print(f"  Tolerance (epsilon_F):         {epsilon_F:.6f}")

# Check if empirical rate is consistent with theoretical bound
if empirical_failure_rate <= delta * 1.5:  # allow some margin for finite sampling
    print(f"\n✓ Validation PASSED: Empirical failure rate ({empirical_failure_rate:.2%})")
    print(f"  is consistent with theoretical bound (delta = {delta:.2%})")
else:
    print(f"\n✗ Validation WARNING: Empirical failure rate ({empirical_failure_rate:.2%})")
    print(f"  exceeds theoretical bound (delta = {delta:.2%})")
    print(f"  This may indicate the bound is not tight or there's an implementation issue.")

print("="*60)


Running 1000 independent experiments with 26492 shots each...
Expected failure rate (delta): 1.0%
Tolerance (epsilon_F): 0.0200



Trials: 100%|██████████| 1000/1000 [21:34<00:00,  1.29s/it]


HOEFFDING BOUND VALIDATION RESULTS
Total trials:                    1000
Shots per trial:                 26492

Bound violations:                0 / 1000
Empirical failure rate:          0.0000 (0.00%)
Theoretical failure rate (delta): 0.0100 (1.00%)

Statistical error statistics:
  Mean error:                    0.000776
  Max error:                     0.000861
  95th percentile error:         0.000861
  99th percentile error:         0.000861
  Tolerance (epsilon_F):         0.020000

✓ Validation PASSED: Empirical failure rate (0.00%)
  is consistent with theoretical bound (delta = 1.00%)


In [ ]:
# ---------------------------------------------------------------------
# 6. Visualization: Plot validation results
# ---------------------------------------------------------------------

import matplotlib.pyplot as plt

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Hoeffding Bound Validation: Statistical Error Distribution', fontsize=16, fontweight='bold')

# Plot 1: Histogram of errors with threshold
ax1 = axes[0, 0]
n_bins = 50
counts, bins, patches = ax1.hist(errors_array, bins=n_bins, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(epsilon_F, color='red', linestyle='--', linewidth=2, label=f'Threshold (ε = {epsilon_F:.4f})')
ax1.axvline(mean_error, color='green', linestyle='--', linewidth=2, label=f'Mean error = {mean_error:.6f}')
ax1.axvline(p95_error, color='orange', linestyle=':', linewidth=2, label=f'95th percentile = {p95_error:.6f}')
ax1.set_xlabel('Statistical Error |F̂ - F_device|', fontsize=11)
ax1.set_ylabel('Frequency', fontsize=11)
ax1.set_title('Distribution of Statistical Errors', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Color bars that exceed threshold
for i, (count, bin_val) in enumerate(zip(counts, bins[:-1])):
    if bin_val > epsilon_F:
        patches[i].set_facecolor('coral')
    else:
        patches[i].set_facecolor('steelblue')

# Plot 2: Cumulative Distribution Function (CDF)
ax2 = axes[0, 1]
sorted_errors = np.sort(errors_array)
cumulative_prob = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors)
ax2.plot(sorted_errors, cumulative_prob, linewidth=2, color='steelblue', label='Empirical CDF')
ax2.axvline(epsilon_F, color='red', linestyle='--', linewidth=2, label=f'Threshold (ε = {epsilon_F:.4f})')
ax2.axhline(1 - delta, color='purple', linestyle='--', linewidth=2, 
            label=f'Theoretical bound (1-δ = {1-delta:.2%})')
ax2.axhline(1 - empirical_failure_rate, color='green', linestyle=':', linewidth=2,
            label=f'Empirical (1-δ_emp = {1-empirical_failure_rate:.2%})')
ax2.set_xlabel('Statistical Error |F̂ - F_device|', fontsize=11)
ax2.set_ylabel('Cumulative Probability', fontsize=11)
ax2.set_title('Cumulative Distribution Function', fontsize=12, fontweight='bold')
ax2.legend(loc='lower right', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1.05])

# Plot 3: Error vs Trial (time series)
ax3 = axes[1, 0]
trial_numbers = np.arange(1, len(errors_array) + 1)
ax3.scatter(trial_numbers, errors_array, alpha=0.5, s=10, color='steelblue', label='Errors')
ax3.axhline(epsilon_F, color='red', linestyle='--', linewidth=2, label=f'Threshold (ε = {epsilon_F:.4f})')
ax3.axhline(mean_error, color='green', linestyle='--', linewidth=1.5, label=f'Mean = {mean_error:.6f}')
violations = np.where(errors_array > epsilon_F)[0]
if len(violations) > 0:
    ax3.scatter(violations + 1, errors_array[violations], color='red', s=30, 
                marker='x', linewidths=2, label=f'Violations ({len(violations)})', zorder=5)
ax3.set_xlabel('Trial Number', fontsize=11)
ax3.set_ylabel('Statistical Error |F̂ - F_device|', fontsize=11)
ax3.set_title('Error Across Trials', fontsize=12, fontweight='bold')
ax3.legend(loc='upper right', fontsize=9)
ax3.grid(True, alpha=0.3)

# Plot 4: Summary statistics bar chart
ax4 = axes[1, 1]
stats_labels = ['Mean', '95th %ile', '99th %ile', 'Max', 'Threshold']
stats_values = [mean_error, p95_error, p99_error, max_error, epsilon_F]
colors = ['steelblue', 'orange', 'darkorange', 'crimson', 'red']
bars = ax4.bar(stats_labels, stats_values, color=colors, alpha=0.7, edgecolor='black')
ax4.set_ylabel('Statistical Error |F̂ - F_device|', fontsize=11)
ax4.set_title('Error Statistics Summary', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, val in zip(bars, stats_values):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
             f'{val:.6f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Add text box with key metrics
textstr = f'Trials: {n_trials}\nShots/trial: {shots}\nViolations: {failures}\nFailure rate: {empirical_failure_rate:.2%}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax4.text(0.02, 0.98, textstr, transform=ax4.transAxes, fontsize=10,
         verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

# Print summary for the plot
print(f"\nVisualization Summary:")
print(f"  • Red dashed line shows the threshold ε = {epsilon_F:.4f}")
print(f"  • {failures} out of {n_trials} trials ({empirical_failure_rate:.2%}) exceeded the threshold")
print(f"  • Theoretical bound predicts ≤ {delta:.2%} failure rate")
